# TrustOCT — Training Notebook 5: Hyperparameter & EDL Optimization Ablations
### Targeted single-variable experiments to optimize the proposed TrustOCT framework
---
**Controlled Single-Variable Experiments**:
- **Baseline**: TrustOCT (`lr=1e-4`, `kl_weight=1.0`, `patience=5`, `kl_annealing=10`)
- **Experiment A**: Lower Learning Rate (`lr=5e-5`)
- **Experiment B**: Reduced KL Loss Weight (`kl_weight=0.3`)
- **Experiment C**: Increased Early Stopping Patience (`patience=8`)
- **Experiment D**: Extended KL Annealing (`kl_annealing_epochs=20`)


In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    !git pull
    print('Repository up to date.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt


In [ ]:
import os, sys, time, shutil, zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print('Running locally or Drive skipped.')


In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print('Downloading Kermany OCT2017 dataset from Kaggle...')
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully!')
    except Exception as e:
        print(f'Download skipped: {e}')
else:
    print('Kermany dataset already present on disk!')


In [ ]:
from src.datasets.dataset import auto_detect_columns, patient_level_split

csv_path = 'kermany_dataset_mapping.csv'
if not os.path.exists(csv_path):
    root_oct = '/content/Kermany/OCT2017 '
    if not os.path.exists(root_oct): root_oct = '/content/Kermany'
    if not os.path.exists(root_oct): root_oct = '/content/OCT2017'
    if os.path.exists(root_oct):
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({'image_path': os.path.join(root, f), 'label': lbl, 'patient_id': pt_id})
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(csv_path, index=False)

df = auto_detect_columns(pd.read_csv(csv_path))
is_colab = os.path.exists('/content')
local_kermany = '/content/Kermany'
local_oct2017 = '/content/OCT2017'
if is_colab and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
    def force_local_path(path_str):
        p = path_str.replace('\\', '/').replace('//', '/')
        parts = p.split('/')
        for folder in ['train', 'val', 'test']:
            if folder in parts:
                idx = parts.index(folder)
                subpath = '/'.join(parts[idx:])
                for candidate in [
                    os.path.join(local_kermany, subpath),
                    os.path.join(local_kermany, 'OCT2017', subpath),
                    os.path.join(local_kermany, 'OCT2017 ', subpath),
                    os.path.join(local_oct2017, subpath)
                ]:
                    if os.path.exists(candidate): return candidate
        return path_str
    df['image_path'] = df['image_path'].apply(force_local_path)

train_df, val_df, test_df = patient_level_split(df)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')


### Experiment A: Lower Learning Rate (`lr = 5e-5`)
Tests whether smoother optimization improves Dirichlet evidence learning.

In [ ]:
from src.training.trainer import run_experiment
print('🚀 Executing Experiment A (lr = 5e-5)...')
run_experiment(
    experiment_id='trustoct_expA',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=5e-5, batch_size=32, device_name=str(device),
    patience=5, kl_weight=1.0, kl_annealing_epochs=10
)


### Experiment B: Reduced KL Loss Weight (`kl_weight = 0.3`)
Tests whether reducing KL penalty allows the classifier to balance accuracy and uncertainty.

In [ ]:
print('🚀 Executing Experiment B (kl_weight = 0.3)...')
run_experiment(
    experiment_id='trustoct_expB',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=1e-4, batch_size=32, device_name=str(device),
    patience=5, kl_weight=0.3, kl_annealing_epochs=10
)


### Experiment C: Increased Early Stopping Patience (`patience = 8`)
Tests whether giving EDL models more epochs allows convergence to a better optimum.

In [ ]:
print('🚀 Executing Experiment C (patience = 8)...')
run_experiment(
    experiment_id='trustoct_expC',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=1e-4, batch_size=32, device_name=str(device),
    patience=8, kl_weight=1.0, kl_annealing_epochs=10
)


### Experiment D: Extended KL Annealing (`kl_annealing_epochs = 20`)
Slows down KL regularization to allow the backbone to form strong features before enforcing uncertainty.

In [ ]:
print('🚀 Executing Experiment D (kl_annealing_epochs = 20)...')
run_experiment(
    experiment_id='trustoct_expD',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=1e-4, batch_size=32, device_name=str(device),
    patience=5, kl_weight=1.0, kl_annealing_epochs=20
)


## Comparative Evaluation & Decision Matrix
Compares Baseline TrustOCT vs Experiments A, B, C, and D.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef, cohen_kappa_score
from src.evaluation.calibration import calculate_ece, calculate_brier_score

CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}
def parse_col(series): return series.apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0))).values

ablation_runs = [
    ('outputs/trustoct/predictions.csv', 'TrustOCT Baseline'),
    ('outputs/trustoct_expA/predictions.csv', 'Exp A (lr = 5e-5)'),
    ('outputs/trustoct_expB/predictions.csv', 'Exp B (kl_weight = 0.3)'),
    ('outputs/trustoct_expC/predictions.csv', 'Exp C (patience = 8)'),
    ('outputs/trustoct_expD/predictions.csv', 'Exp D (annealing = 20)')
]

rows = []
for path, display_name in ablation_runs:
    if os.path.exists(path):
        df_p = pd.read_csv(path)
        y_true = parse_col(df_p['true_label'])
        y_pred = parse_col(df_p['pred_label'])
        probs = df_p[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
        u_arr = df_p['uncertainty'].values if 'uncertainty' in df_p.columns else None
        
        acc = accuracy_score(y_true, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        kappa = cohen_kappa_score(y_true, y_pred)
        
        conf = (1.0 - u_arr) * np.max(probs, axis=1) if u_arr is not None else np.max(probs, axis=1)
        ece = calculate_ece(conf, (y_pred == y_true).astype(int))
        brier = calculate_brier_score(probs, y_true)
        
        rows.append({
            'Experiment': display_name,
            'Accuracy (%)': f"{acc*100:.2f}%",
            'Macro F1': f"{f1:.4f}",
            'Kappa': f"{kappa:.4f}",
            'ECE (Calibration) ↓': f"{ece:.4f}",
            'Brier Score ↓': f"{brier:.4f}"
        })

if len(rows) > 0:
    matrix_df = pd.DataFrame(rows)
    print('\n--- TRUSTOCT HYPERPARAMETER ABLATION DECISION MATRIX ---')
    display(matrix_df)
    os.makedirs('results/tables', exist_ok=True)
    matrix_df.to_csv('results/tables/trustoct_ablation_decision_matrix.csv', index=False)

# Sync to Drive
shared_folder = '/content/drive/MyDrive/TrustOCT_Results/outputs'
os.makedirs(shared_folder, exist_ok=True)
for exp_id in ['trustoct_expA', 'trustoct_expB', 'trustoct_expC', 'trustoct_expD']:
    src = f'outputs/{exp_id}'
    dst = os.path.join(shared_folder, exp_id)
    if os.path.exists(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Synced {exp_id} to shared Google Drive.')
